In [0]:
# =============================================================================
# Translation Workflow with Back-Translation + Judge
# =============================================================================
# This notebook demonstrates a production-ready translation pipeline for PDF 
# documents stored in a Unity Catalog Volume, using Databricks AI Functions.
#
# Architecture Overview:
# ┌─────────────┐    ┌─────────────┐    ┌──────────────┐    ┌─────────────┐    ┌─────────────┐
# │  PDF Docs   │───▶│  Parse PDF  │───▶│  Translate   │───▶│   Back      │───▶│   Judge     │
# │  (Volume)   │    │  (ai_parse  │    │  to English  │    │  Translate  │    │  (ai_query) │
# │             │    │  _document) │    │  (ai_query)  │    │  (ai_query) │    │             │
# └─────────────┘    └─────────────┘    └──────────────┘    └─────────────┘    └─────────────┘
#                                                                                      │
#                                                                               ┌──────▼──────┐
#                                                                               │  Final      │
#                                                                               │  Output     │
#                                                                               │  (Delta)    │
#                                                                               └─────────────┘
#
# Pipeline Steps:
# 1. Ingest PDFs from Volume using READ_FILES + ai_parse_document
# 2. Detect source language (auto-detection via ai_query)
# 3. Translate extracted text to English (ai_query with translation prompt)
# 4. Back-translate English text to original language (ai_query)
# 5. Judge quality by comparing original vs back-translation (ai_query with scoring prompt)
# 6. Flag low-quality translations for human review
#
# All steps use Databricks AI Functions for scalability and simplicity.

print("Translation Workflow Architecture - see comments above for diagram")

In [0]:
# =============================================================================
# STEP 1: Configuration
# =============================================================================
# Update these variables to match your environment

# Source volume containing PDFs
SOURCE_VOLUME = "/Volumes/my_catalog/my_schema/pdf_documents"

# Output catalog/schema for the pipeline tables
OUTPUT_CATALOG = "my_catalog"
OUTPUT_SCHEMA = "my_schema"

# LLM endpoint for translation and judging 
# Use a Databricks-hosted foundation model (e.g., Claude, Meta Llama, etc.)
LLM_ENDPOINT = "databricks-claude-sonnet-4"  # or any FMAPI endpoint

# Quality threshold: translations scoring below this are flagged for human review
QUALITY_THRESHOLD = 7  # on a 1-10 scale

print(f"Source Volume: {SOURCE_VOLUME}")
print(f"Output: {OUTPUT_CATALOG}.{OUTPUT_SCHEMA}")
print(f"LLM Endpoint: {LLM_ENDPOINT}")
print(f"Quality Threshold: {QUALITY_THRESHOLD}/10")

In [0]:
# =============================================================================
# STEP 2: Parse PDFs from Volume
# =============================================================================
# This step reads binary PDF files from the volume and extracts text content
# using ai_parse_document(). The function returns structured elements (text,
# tables, figures, headers) with page-level metadata.

parse_pdf_sql = f"""
CREATE OR REPLACE TABLE {OUTPUT_CATALOG}.{OUTPUT_SCHEMA}.parsed_documents AS
WITH raw_files AS (
  SELECT 
    path,
    content
  FROM READ_FILES(
    '{SOURCE_VOLUME}/*.pdf', 
    format => 'binaryFile'
  )
),
parsed AS (
  SELECT
    path AS source_path,
    ai_parse_document(content) AS parsed_result
  FROM raw_files
)
SELECT
  source_path,
  -- Extract file name for easier reference
  regexp_extract(source_path, '([^/]+)$', 1) AS file_name,
  -- Get all text elements concatenated per document
  concat_ws(
    '\n\n',
    transform(
      filter(parsed_result:document:elements, x -> x:type IN ('text', 'title', 'section_header')),
      x -> x:content::STRING
    )
  ) AS extracted_text,
  -- Keep structured elements for granular processing
  parsed_result:document:elements AS elements,
  -- Number of pages
  parsed_result:document:num_pages AS num_pages,
  current_timestamp() AS parsed_at
FROM parsed
"""

print("SQL for Step 2 - Parse PDFs:")
print(parse_pdf_sql)

In [0]:
# =============================================================================
# STEP 3: Detect Source Language
# =============================================================================
# Before translating, we detect the source language of each document.
# This is essential for the back-translation step (we need to know what 
# language to back-translate INTO).

detect_language_sql = f"""
CREATE OR REPLACE TABLE {OUTPUT_CATALOG}.{OUTPUT_SCHEMA}.documents_with_language AS
SELECT
  source_path,
  file_name,
  extracted_text,
  elements,
  num_pages,
  parsed_at,
  ai_query(
    '{LLM_ENDPOINT}',
    CONCAT(
      'Identify the primary language of the following text. ',
      'Return ONLY the ISO 639-1 language code (e.g., "fr" for French, "de" for German, ',
      '"ja" for Japanese, "zh" for Chinese, "es" for Spanish, "pt" for Portuguese, ',
      '"ko" for Korean, "ar" for Arabic, "ru" for Russian, "it" for Italian, "nl" for Dutch). ',
      'If the text is already in English, return "en". ',
      'Text: ', LEFT(extracted_text, 2000)
    ),
    responseFormat => 'TEXT'
  ) AS detected_language
FROM {OUTPUT_CATALOG}.{OUTPUT_SCHEMA}.parsed_documents
"""

print("SQL for Step 3 - Detect Source Language:")
print(detect_language_sql)

In [0]:
# =============================================================================
# STEP 4: Translate to English (Forward Translation)
# =============================================================================
# For documents NOT already in English, perform the forward translation.
# We use ai_query with a detailed prompt to ensure high-quality translation.
# 
# NOTE: For long documents, you may want to chunk text and translate per-chunk.
# The example below processes documents in chunks of ~3000 chars for reliability.

translate_to_english_sql = f"""
CREATE OR REPLACE TABLE {OUTPUT_CATALOG}.{OUTPUT_SCHEMA}.translated_documents AS
SELECT
  source_path,
  file_name,
  extracted_text AS original_text,
  detected_language,
  num_pages,
  parsed_at,
  CASE 
    WHEN TRIM(detected_language) = 'en' THEN extracted_text  -- Already English
    ELSE ai_query(
      '{LLM_ENDPOINT}',
      CONCAT(
        'You are a professional translator. Translate the following text to English. ',
        'Maintain the original meaning, tone, and structure as closely as possible. ',
        'Preserve any technical terminology. Do not add explanations or notes. ',
        'Only output the translated text.\n\n',
        'Source language: ', detected_language, '\n\n',
        'Text to translate:\n', extracted_text
      ),
      responseFormat => 'TEXT'
    )
  END AS english_translation,
  current_timestamp() AS translated_at
FROM {OUTPUT_CATALOG}.{OUTPUT_SCHEMA}.documents_with_language
"""

print("SQL for Step 4 - Forward Translation to English:")
print(translate_to_english_sql)

In [0]:
# =============================================================================
# STEP 5: Back-Translation (English → Original Language)
# =============================================================================
# Back-translate the English translation back to the source language.
# This creates a "round-trip" that allows us to assess translation quality
# by comparing the back-translation with the original text.

back_translate_sql = f"""
CREATE OR REPLACE TABLE {OUTPUT_CATALOG}.{OUTPUT_SCHEMA}.back_translated_documents AS
SELECT
  source_path,
  file_name,
  original_text,
  detected_language,
  english_translation,
  num_pages,
  translated_at,
  CASE 
    WHEN TRIM(detected_language) = 'en' THEN english_translation  -- No back-translation needed
    ELSE ai_query(
      '{LLM_ENDPOINT}',
      CONCAT(
        'You are a professional translator. Translate the following English text to ',
        detected_language, '. ',
        'Maintain the original meaning, tone, and structure as closely as possible. ',
        'Preserve any technical terminology. Do not add explanations or notes. ',
        'Only output the translated text.\n\n',
        'Text to translate:\n', english_translation
      ),
      responseFormat => 'TEXT'
    )
  END AS back_translation,
  current_timestamp() AS back_translated_at
FROM {OUTPUT_CATALOG}.{OUTPUT_SCHEMA}.translated_documents
"""

print("SQL for Step 5 - Back-Translation:")
print(back_translate_sql)

In [0]:
# =============================================================================
# STEP 6: LLM Judge - Quality Assessment
# =============================================================================
# The Judge compares the ORIGINAL text with the BACK-TRANSLATION to assess 
# translation quality. A high-quality translation should produce a back-translation
# that is semantically very close to the original.
#
# The judge provides:
# - A quality score (1-10)
# - Specific issues identified (omissions, mistranslations, tone shifts)
# - A recommendation (ACCEPT / REVISE / REJECT)

judge_sql = f"""
CREATE OR REPLACE TABLE {OUTPUT_CATALOG}.{OUTPUT_SCHEMA}.translation_judged AS
SELECT
  source_path,
  file_name,
  original_text,
  detected_language,
  english_translation,
  back_translation,
  num_pages,
  translated_at,
  back_translated_at,
  CASE 
    WHEN TRIM(detected_language) = 'en' THEN NULL  -- No judging needed for English docs
    ELSE ai_query(
      '{LLM_ENDPOINT}',
      CONCAT(
        'You are an expert translation quality judge. Your task is to assess the quality ',
        'of a translation by comparing the original text with its back-translation.\n\n',
        'A good translation will produce a back-translation that preserves the meaning, ',
        'key information, and tone of the original. Minor stylistic differences are acceptable.\n\n',
        'ORIGINAL TEXT (in ', detected_language, '):\n',
        LEFT(original_text, 3000), '\n\n',
        'BACK-TRANSLATION (from English back to ', detected_language, '):\n',
        LEFT(back_translation, 3000), '\n\n',
        'Evaluate the translation quality and respond in the following JSON format ONLY:\n',
        '{{\n',
        '  "quality_score": <1-10 integer>,\n',
        '  "semantic_preservation": <1-10>,\n',
        '  "completeness": <1-10>,\n',
        '  "issues": ["<issue1>", "<issue2>", ...],\n',
        '  "recommendation": "<ACCEPT|REVISE|REJECT>",\n',
        '  "explanation": "<brief explanation>"\n',
        '}}'
      ),
      responseFormat => 'JSON'
    )
  END AS judge_result,
  current_timestamp() AS judged_at
FROM {OUTPUT_CATALOG}.{OUTPUT_SCHEMA}.back_translated_documents
"""

print("SQL for Step 6 - LLM Judge Quality Assessment:")
print(judge_sql)

In [0]:
# =============================================================================
# STEP 7: Final Output Table with Quality Flags
# =============================================================================
# Create the final output table that extracts judge scores and flags documents
# needing human review.

final_output_sql = f"""
CREATE OR REPLACE TABLE {OUTPUT_CATALOG}.{OUTPUT_SCHEMA}.translation_final AS
SELECT
  source_path,
  file_name,
  detected_language,
  english_translation,
  num_pages,
  translated_at,
  -- Parse judge results
  judge_result:quality_score::INT AS quality_score,
  judge_result:semantic_preservation::INT AS semantic_preservation_score,
  judge_result:completeness::INT AS completeness_score,
  judge_result:recommendation::STRING AS recommendation,
  judge_result:explanation::STRING AS judge_explanation,
  judge_result:issues AS issues,
  -- Flag for human review
  CASE
    WHEN TRIM(detected_language) = 'en' THEN 'SKIPPED'
    WHEN judge_result:quality_score::INT >= {QUALITY_THRESHOLD} THEN 'APPROVED'
    WHEN judge_result:recommendation::STRING = 'REJECT' THEN 'REJECTED'
    ELSE 'NEEDS_REVIEW'
  END AS status,
  judged_at
FROM {OUTPUT_CATALOG}.{OUTPUT_SCHEMA}.translation_judged
"""

print("SQL for Step 7 - Final Output with Quality Flags:")
print(final_output_sql)

In [0]:
# =============================================================================
# ALTERNATIVE: Spark Declarative Pipeline (SDP) Implementation
# =============================================================================
# For production, this is best implemented as a Lakeflow Spark Declarative Pipeline.
# Below is the equivalent SDP code that handles incremental processing.

sdp_code = '''
-- =============================================================================
-- Spark Declarative Pipeline: Translation Workflow with Back-Translation + Judge
-- =============================================================================

-- Step 1: Ingest and parse PDFs (streaming from volume)
CREATE OR REFRESH STREAMING TABLE parsed_documents AS
SELECT
  path AS source_path,
  regexp_extract(path, '([^/]+)$', 1) AS file_name,
  ai_parse_document(content) AS parsed_result,
  current_timestamp() AS ingested_at
FROM STREAM READ_FILES(
  '/Volumes/my_catalog/my_schema/pdf_documents/*.pdf',
  format => 'binaryFile'
);

-- Step 2: Extract text and detect language
CREATE OR REFRESH MATERIALIZED VIEW documents_with_text AS
SELECT
  source_path,
  file_name,
  concat_ws(
    '\n\n',
    transform(
      filter(parsed_result:document:elements, x -> x:type IN ('text', 'title', 'section_header')),
      x -> x:content::STRING
    )
  ) AS extracted_text,
  parsed_result:document:num_pages AS num_pages,
  ai_query(
    'databricks-claude-sonnet-4',
    CONCAT(
      'Identify the primary language. Return ONLY the ISO 639-1 code. Text: ',
      LEFT(
        concat_ws(' ', transform(
          filter(parsed_result:document:elements, x -> x:type = 'text'),
          x -> x:content::STRING
        )),
        1000
      )
    ),
    responseFormat => 'TEXT'
  ) AS detected_language,
  ingested_at
FROM LIVE.parsed_documents;

-- Step 3: Forward translation to English
CREATE OR REFRESH MATERIALIZED VIEW translated_to_english AS
SELECT
  source_path,
  file_name,
  extracted_text AS original_text,
  detected_language,
  num_pages,
  CASE 
    WHEN TRIM(detected_language) = 'en' THEN extracted_text
    ELSE ai_query(
      'databricks-claude-sonnet-4',
      CONCAT(
        'You are a professional translator. Translate the following text to English. ',
        'Maintain the original meaning, tone, and structure. ',
        'Only output the translated text.\n\nSource language: ', detected_language,
        '\n\nText:\n', extracted_text
      ),
      responseFormat => 'TEXT'
    )
  END AS english_translation,
  ingested_at
FROM LIVE.documents_with_text;

-- Step 4: Back-translation (English → original language)
CREATE OR REFRESH MATERIALIZED VIEW back_translated AS
SELECT
  *,
  CASE 
    WHEN TRIM(detected_language) = 'en' THEN english_translation
    ELSE ai_query(
      'databricks-claude-sonnet-4',
      CONCAT(
        'You are a professional translator. Translate to ', detected_language,
        '. Only output the translated text.\n\nText:\n', english_translation
      ),
      responseFormat => 'TEXT'
    )
  END AS back_translation
FROM LIVE.translated_to_english;

-- Step 5: Judge evaluates quality via original vs. back-translation comparison
CREATE OR REFRESH MATERIALIZED VIEW translation_judged AS
SELECT
  source_path,
  file_name,
  original_text,
  detected_language,
  english_translation,
  back_translation,
  num_pages,
  CASE 
    WHEN TRIM(detected_language) = 'en' THEN NULL
    ELSE ai_query(
      'databricks-claude-sonnet-4',
      CONCAT(
        'You are a translation quality judge. Compare the original with the back-translation. ',
        'Score 1-10. Respond as JSON: {"quality_score":<int>,"recommendation":"ACCEPT|REVISE|REJECT","issues":[...],"explanation":"..."}\n\n',
        'ORIGINAL (', detected_language, '):\n', LEFT(original_text, 3000),
        '\n\nBACK-TRANSLATION:\n', LEFT(back_translation, 3000)
      ),
      responseFormat => 'JSON'
    )
  END AS judge_result,
  ingested_at
FROM LIVE.back_translated;

-- Step 6: Final output with status flags
CREATE OR REFRESH MATERIALIZED VIEW translation_final AS
SELECT
  source_path,
  file_name,
  detected_language,
  english_translation,
  num_pages,
  judge_result:quality_score::INT AS quality_score,
  judge_result:recommendation::STRING AS recommendation,
  judge_result:explanation::STRING AS explanation,
  CASE
    WHEN TRIM(detected_language) = 'en' THEN 'SKIPPED'
    WHEN judge_result:quality_score::INT >= 7 THEN 'APPROVED'
    WHEN judge_result:recommendation::STRING = 'REJECT' THEN 'REJECTED'
    ELSE 'NEEDS_REVIEW'
  END AS status,
  ingested_at
FROM LIVE.translation_judged;
'''

print("Spark Declarative Pipeline (SDP) SQL code generated successfully.")
print("This pipeline processes PDFs incrementally as they arrive in the volume.")

In [0]:
# =============================================================================
# STEP 8: Chunking Strategy for Long Documents
# =============================================================================
# For documents that exceed LLM context limits, implement a chunking strategy.
# This ensures reliable translation of long PDFs by processing paragraph-level
# elements individually, then reassembling.

chunking_approach = """
-- Explode document elements into individual rows for per-element translation
-- This handles long documents that would exceed context windows

CREATE OR REPLACE TABLE my_catalog.my_schema.document_chunks AS
SELECT
  source_path,
  file_name,
  detected_language,
  elem.id AS element_id,
  elem.type AS element_type,
  elem.content::STRING AS chunk_text,
  elem.metadata:page_number::INT AS page_number
FROM my_catalog.my_schema.documents_with_language
LATERAL VIEW EXPLODE(
  filter(elements, x -> x:type IN ('text', 'title', 'section_header', 'table'))
) AS elem
WHERE LENGTH(elem.content::STRING) > 10;  -- Skip trivially short elements

-- Translate each chunk independently
CREATE OR REPLACE TABLE my_catalog.my_schema.translated_chunks AS
SELECT
  source_path,
  file_name,
  element_id,
  element_type,
  page_number,
  chunk_text AS original_chunk,
  detected_language,
  CASE
    WHEN TRIM(detected_language) = 'en' THEN chunk_text
    ELSE ai_query(
      'databricks-claude-sonnet-4',
      CONCAT(
        'Translate to English. Preserve formatting and meaning. Only output the translation.\n\n',
        chunk_text
      ),
      responseFormat => 'TEXT'
    )
  END AS translated_chunk
FROM my_catalog.my_schema.document_chunks;

-- Reassemble translated chunks into full documents
CREATE OR REPLACE TABLE my_catalog.my_schema.reassembled_translations AS
SELECT
  source_path,
  file_name,
  detected_language,
  concat_ws(
    '\n\n',
    collect_list(translated_chunk)
  ) AS english_translation
FROM my_catalog.my_schema.translated_chunks
GROUP BY source_path, file_name, detected_language
ORDER BY source_path, element_id;
"""

print("Chunking Strategy for Long Documents:")
print(chunking_approach)

In [0]:

# =============================================================================
# STEP 9: Retry Logic for Failed/Low-Quality Translations
# =============================================================================
# Documents flagged as NEEDS_REVIEW or REJECTED can be retried with a different
# strategy: use a different model, provide more context, or use domain-specific
# terminology glossaries.

retry_logic = """
-- Retry translations that failed quality assessment
-- Strategy: Use additional context and request the model to focus on the flagged issues

CREATE OR REPLACE TABLE my_catalog.my_schema.translation_retries AS
SELECT
  t.source_path,
  t.file_name,
  t.original_text,
  t.detected_language,
  t.english_translation AS first_attempt,
  f.quality_score AS first_score,
  f.explanation AS first_issues,
  ai_query(
    'databricks-claude-sonnet-4',
    CONCAT(
      'You are a professional translator. A previous translation attempt had quality issues: ',
      f.explanation,
      '\\n\\nPlease provide an improved English translation that addresses these issues. ',
      'Focus on accuracy and completeness.\\n\\n',
      'Original text (', t.detected_language, '):\\n', t.original_text
    ),
    responseFormat => 'TEXT'
  ) AS improved_translation,
  2 AS attempt_number,
  current_timestamp() AS retried_at
FROM my_catalog.my_schema.back_translated_documents t
JOIN my_catalog.my_schema.translation_final f
  ON t.source_path = f.source_path
WHERE f.status IN ('NEEDS_REVIEW', 'REJECTED');
"""

print("Retry Logic for Low-Quality Translations:")
print(retry_logic)


In [0]:

# =============================================================================
# SUMMARY: Deployment Options
# =============================================================================

summary = """
╔══════════════════════════════════════════════════════════════════════════════╗
║          TRANSLATION WORKFLOW - DEPLOYMENT OPTIONS                          ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  OPTION A: Spark Declarative Pipeline (RECOMMENDED for production)           ║
║  ─────────────────────────────────────────────────────────────────           ║
║  • Incremental processing as new PDFs arrive                                 ║
║  • Automatic retry and error handling                                        ║
║  • Built-in lineage tracking                                                 ║
║  • Easy monitoring via Pipeline UI                                           ║
║  • Deploy: Create SDP pipeline with the SQL from Step 8                      ║
║                                                                              ║
║  OPTION B: Databricks Workflow (for scheduled batch runs)                     ║
║  ─────────────────────────────────────────────────────────────────           ║
║  • Run Steps 2-7 as sequential notebook tasks                                ║
║  • Use If/Else tasks for retry logic                                         ║
║  • Schedule daily/hourly based on volume arrival                             ║
║  • Add alert notifications for REJECTED translations                         ║
║                                                                              ║
║  OPTION C: Single Notebook (for ad-hoc/testing)                              ║
║  ─────────────────────────────────────────────────────────────────           ║
║  • Run all steps sequentially in one notebook                                ║
║  • Good for prototyping and parameter tuning                                 ║
║  • Use with serverless compute for simplicity                                ║
║                                                                              ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  KEY DATABRICKS FEATURES USED:                                               ║
║  • ai_parse_document() - Extract text from PDFs (supports PDF, DOCX, etc.)   ║
║  • ai_query()          - General-purpose LLM for translation & judging       ║
║  • ai_translate()      - Quick translations (8 languages, no custom prompt)  ║
║  • READ_FILES()        - Read binary files from Unity Catalog Volumes         ║
║  • Serverless compute  - Auto-scaling, no cluster management                 ║
║  • Unity Catalog       - Governance, lineage, access control                 ║
║                                                                              ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  COST CONSIDERATIONS:                                                        ║
║  • ai_parse_document: Billed per page parsed                                 ║
║  • ai_query: Billed per token (input + output)                               ║
║  • Each document goes through 4 LLM calls minimum:                           ║
║    1. Language detection  (~small input)                                      ║
║    2. Forward translation (full document)                                     ║
║    3. Back-translation    (full document)                                     ║
║    4. Judge evaluation    (original + back-translation)                       ║
║  • Optimization: Skip back-translation + judge for documents already         ║
║    in English (detected_language = 'en')                                     ║
║  • Optimization: Use ai_translate() for supported languages (cheaper)        ║
║    and ai_query() only for unsupported languages                             ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

print(summary)
